In [1]:
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("rmisra/news-category-dataset")

df = pd.read_json(path + '/News_Category_Dataset_v3.json', lines=True)

print(df.shape)
print("Path to dataset files:", path)

(209527, 6)
Path to dataset files: /home/jens/.cache/kagglehub/datasets/rmisra/news-category-dataset/versions/3


In [2]:
df.describe()

,date
count,209527
mean,2015-04-30 00:44:14.344308
min,2012-01-28 00:00:00
25%,2013-08-10 00:00:00
50%,2015-03-16 00:00:00
75%,2016-11-01 00:00:00
max,2022-09-23 00:00:00


In [3]:
df.sample(5, random_state=42)

,link,headline,category,short_description,authors,date
128310,https://www.huffingtonpost.com/entry/what-if-w...,What If We Were All Family Generation Changers?,IMPACT,"What if, in doing so, we won't just create new...","Matt Murrie, ContributorEdupreneur, Cofounder/...",2014-06-20
139983,https://www.huffingtonpost.comhttp://www.washi...,Firestorm At AOL Over Employee Benefit Cuts,BUSINESS,It should have been a glorious week for AOL ch...,,2014-02-08
42339,https://www.huffingtonpost.com/entry/time-runs...,Dakota Access Protesters Arrested As Deadline ...,POLITICS,A few protesters who refused to leave remained...,"Michael McLaughlin & Josh Morgan, The Huffingt...",2017-02-22
131494,https://www.huffingtonpost.com/entry/one-glimp...,One Glimpse Of These Baby Kit Foxes And You'll...,GREEN,,,2014-05-14
163649,https://www.huffingtonpost.com/entry/mens-swea...,"Mens' Sweat Pheromone, Androstadienone, Influe...",SCIENCE,Scientists didn't know if humans played that g...,Melissa Cronin,2013-06-02


In [4]:
print("Number of unique categories:", df['category'].nunique())
print("\nCategory counts:")
print(df['category'].value_counts())

Number of unique categories: 42

Category counts:
category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATIN

In [5]:
df['combined_text'] = df['headline'] + ' ' + df['short_description']

# Clean dataset
print(df.shape)
clean_df = df[['combined_text', 'category']].copy()
clean_df.dropna(inplace=True)
clean_df.drop_duplicates(inplace=True)
print(clean_df.shape)
clean_df.head(5)

(209527, 7)
(209056, 2)


,combined_text,category
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY
3,The Funniest Tweets From Parents This Week (Se...,PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS


In [6]:
# BERT handles its own tokenization — no need for manual preprocessing
# We only do minimal cleaning: strip leading/trailing whitespace
clean_df['combined_text'] = clean_df['combined_text'].str.strip()
clean_df.head(5)

,combined_text,category
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY
3,The Funniest Tweets From Parents This Week (Se...,PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(clean_df['category'])

x = clean_df['combined_text'].values
y = y_encoded

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("Classes:", label_encoder.classes_)
print("Train size:", len(x_train))
print("Test size:", len(x_test))

Classes: ['ARTS' 'ARTS & CULTURE' 'BLACK VOICES' 'BUSINESS' 'COLLEGE' 'COMEDY'
 'CRIME' 'CULTURE & ARTS' 'DIVORCE' 'EDUCATION' 'ENTERTAINMENT'
 'ENVIRONMENT' 'FIFTY' 'FOOD & DRINK' 'GOOD NEWS' 'GREEN' 'HEALTHY LIVING'
 'HOME & LIVING' 'IMPACT' 'LATINO VOICES' 'MEDIA' 'MONEY' 'PARENTING'
 'PARENTS' 'POLITICS' 'QUEER VOICES' 'RELIGION' 'SCIENCE' 'SPORTS' 'STYLE'
 'STYLE & BEAUTY' 'TASTE' 'TECH' 'THE WORLDPOST' 'TRAVEL' 'U.S. NEWS'
 'WEDDINGS' 'WEIRD NEWS' 'WELLNESS' 'WOMEN' 'WORLD NEWS' 'WORLDPOST']
Train size: 167244
Test size: 41812


In [8]:
# Install required libraries if not already installed
# !pip install transformers torch datasets

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Training on CPU
device = torch.device('cpu')
print("Using device:", device)

Using device: cpu


In [10]:
# Load the BERT tokenizer
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 16   # Smaller batch size is easier on CPU memory
EPOCHS = 3
LEARNING_RATE = 2e-5
NUM_CLASSES = len(label_encoder.classes_)

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded.")
print("Number of classes:", NUM_CLASSES)

Tokenizer loaded.
Number of classes: 42


In [11]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = NewsDataset(x_train, y_train, tokenizer, MAX_LEN)
test_dataset  = NewsDataset(x_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Test batches: ", len(test_loader))

Train batches: 10453
Test batches:  2614


In [12]:
# Load pre-trained BERT with a classification head
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES
)
print("Model loaded on CPU")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on CPU


In [13]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print("Total training steps:", total_steps)

Total training steps: 31359


In [14]:
import time

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    start = time.time()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels         = batch['label']

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        if (step + 1) % 100 == 0:
            elapsed = time.time() - start
            print(f"  Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | "
                  f"Loss: {total_loss / (step+1):.4f} | Elapsed: {elapsed:.1f}s")

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} complete — Avg Loss: {avg_loss:.4f}\n")

  Epoch 1 | Step 100/10453 | Loss: 3.7469 | Elapsed: 330.3s
  Epoch 1 | Step 200/10453 | Loss: 3.7056 | Elapsed: 595.0s


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels         = batch['label']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

y_pred = all_preds
y_test_eval = all_labels

accuracy  = accuracy_score(y_test_eval, y_pred)
precision = precision_score(y_test_eval, y_pred, average='weighted')
recall    = recall_score(y_test_eval, y_pred, average='weighted')
f1        = f1_score(y_test_eval, y_pred, average='weighted')

print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-score:  {f1 * 100:.2f}%")

print("\nClassification Report:\n",
      classification_report(y_test_eval, y_pred, target_names=label_encoder.classes_))

In [ ]:
import joblib
import os

# Save the BERT model and tokenizer
model_save_path = f'bert_category_model_acc_{accuracy * 100:.2f}%'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print("BERT model saved to:", model_save_path)

# Save the label encoder
joblib.dump(label_encoder, 'category_label_encoder.pkl')
print("Label encoder saved.")

In [ ]:
# Inference helper — classify new headlines

def predict_category(text, model, tokenizer, label_encoder, max_len=128):
    model.eval()
    encoding = tokenizer(
        text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids']
    attention_mask = encoding['attention_mask']

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred    = torch.argmax(outputs.logits, dim=1).item()

    return label_encoder.inverse_transform([pred])[0]


# Example predictions
examples = [
    "Stock market hits record high amid strong earnings reports",
    "New study reveals benefits of Mediterranean diet for heart health",
    "Local football team wins championship in overtime thriller"
]

for text in examples:
    category = predict_category(text, model, tokenizer, label_encoder)
    print(f"Text:     {text}")
    print(f"Category: {category}\n")